# TP1 Modèles de langue: exploration de la tokenization et des embeddings

Dans ce TP nous allons explorer un peu la tokenization d'un modèle de langue, en regardant par exemple la correspondance avec les mots d'un dictionnaire classique.

Puis nous allons étudier l'espace de représentation des embeddings associés aux sous-tokens dans un modèle, et tester si les mots proches en sens sont tokenisés avec des sous-tokens proches.

Enfin, nous visualiserons les correspondances entre 2 langues dans l'espace des représentations créé par un modèle multilingue.

In [37]:
%%capture
# pour faire de la réduction de dimension de l'espace des embeddings
!pip install umap-learn
# dictionnaire multilingue de traductions mot -> mot
!pip install word2word

In [38]:
# lexique de mots anglais
!wget https://www.irit.fr/~Philippe.Muller/lexicon.txt.gz
!gunzip lexicon.txt.gz

--2026-09-17 16:18:31--  https://www.irit.fr/~Philippe.Muller/lexicon.txt.gz
Résolution de www.irit.fr (www.irit.fr)… 141.115.28.2
Connexion à www.irit.fr (www.irit.fr)|141.115.28.2|:443… connecté.
requête HTTP transmise, en attente de la réponse… 200 OK
Taille : 372728 (364K) [application/x-gzip]
Sauvegarde en : « lexicon.txt.gz »

lexicon.txt.gz      100%[===================>] 363,99K  --.-KB/s    ds 0,1s    

2026-09-17 16:18:32 (2,81 MB/s) — « lexicon.txt.gz » sauvegardé [372728/372728]

gzip: lexicon.txt already exists; do you wish to overwrite (y or n)? ^C


In [1]:
import numpy as np
import umap
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import pandas as pds

/home/klk4720a/Téléchargements/TAL/TP1/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForMaskedLM
import torch
import pandas as pds
import itertools
import gzip

In [3]:
import word2word

In [4]:
# on peut essayer différents modèles, mais seul le dernier est multilingue
#MODEL = "bert-base-uncased"
#MODEL = "roberta-base"
MODEL = "bert-base-multilingual-cased"
cpt = {}

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [48]:
lexicon = set([x.strip() for x in open("lexicon.txt").readlines()])

print(lexicon)

{'betterness', 'gloomth', 'sacrate', 'conciliate', 'nefarious', 'dexterity', 'artificious', 'water-furrow', 'dipterous', 'water course', 'oke', 'proxene', 'anconeus', 'shemite', 'cryptographist', 'electro-chemical', 'hypapophysis', 'elwand', 'equitation', 'pinesap', 'gaffer', 'stereoscopist', 'cowage', 'unreserved', 'acanthopterous', 'fluvialist', 'potential', 'mesityl', 'sensitivity', 'impeaching', 'minos', 'polygynous', 'pedestrianize', 'rostrated', 'serrifera', 'preternaturalism', 'sagittocyst', 'blackthorn', 'piend', 'hoaxed', 'disrespective', 'bowmen', 'hyemation', 'parabolism', 'stanzaic', 'watchman', 'summons', 'exarillate', 'devolvement', 'foster', 'hogsties', 'sketchbook', 'declined', 'succinamate', 'inobservation', 'angulated', 'otitis', 'humiliating', 'trending', 'cartographically', 'crescendo', 'boule', 'disconsecrate', 'hymnody', 'samian', 'inexpansible', 'inconsistently', 'spiranthy', 'principally', 'manitu', 'usuring', 'areometer', 'deplication', 'shipping', 'disinterrin

# Tokenization

- Appliquer le tokenizer associé à des modèles listés plus haut, à tous les mots du lexique anglais, pour récupérer tous les sous-mots utilisés, et faites un comptage du nombre de mots dans lequel chacun apparait.

Que peut-on dire des plus fréquents ?

(conseils: le faire en un seul appel au tokenizer, pas mot à mot; le comptage peut être fait avec le module collections.Counter de python ou bien pandas)

- A quoi correspondent les symboles spéciaux ?

- Créer un dictionnaire qui associe à chaque mot du lexique la liste des sous-tokens qui le composent
(pour être plus efficace, vous pouvez prendre la liste des sous-mots générés à la 1er question; la 2e question est important aussi !)


In [71]:
from collections import Counter
token_info=tokenizer.tokenize(" ".join(list(lexicon)))
cpt = (Counter(token_info))

In [72]:
cpt.most_common(20)

[('-', 3445),
 ('##d', 2873),
 ('##y', 2564),
 ('##e', 2445),
 ('##ly', 2014),
 ('##ing', 1993),
 ('##s', 1942),
 ('##ed', 1927),
 ('##l', 1712),
 ('##r', 1635),
 ('in', 1465),
 ('##t', 1462),
 ('##us', 1452),
 ('##te', 1392),
 ('s', 1343),
 ('##er', 1330),
 ('##c', 1210),
 ('c', 1204),
 ('##ness', 1199),
 ('##n', 1167)]

# Analyse des embeddings de base préentrainés

Charger un des modèles avec la library transformers avec AutoModelForMaskedLM.

Affichez la structure du modèle pour trouvez la matrice d'embeddings des sous tokens du vocabulaire du modèle.

Quelle est sa taille ? Quelle est la dimension des embeddings ?

In [8]:
model = AutoModelForMaskedLM.from_pretrained(MODEL)

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1193.14it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
model

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, e

In [40]:
model.bert.embeddings.word_embeddings.__dict__['_parameters']['weight']

Parameter containing:
tensor([[ 0.0260, -0.0062, -0.0041,  ...,  0.0297,  0.0242,  0.0197],
        [ 0.0104, -0.0136,  0.0067,  ...,  0.0124,  0.0267,  0.0337],
        [ 0.0221, -0.0036,  0.0193,  ...,  0.0069,  0.0268,  0.0050],
        ...,
        [ 0.0068,  0.0189,  0.0267,  ...,  0.0229,  0.0647,  0.0437],
        [ 0.0184,  0.0148,  0.0243,  ...,  0.0321,  0.0071,  0.0204],
        [ 0.0214,  0.0188, -0.0134,  ..., -0.0060,  0.0058, -0.0059]],
       requires_grad=True)

### on a 119547 lignes et 768 colones dans la matrice d'embedding
### Taille: 119547*768
### Dim: 768

## Similarité de mots

- Testez la similarité de deux sous-tokens qui sont aussi des mots (pour faire plus simple), en utilisant la similarité cosinus (dans torch.nn) entre leur vecteur d'embedding de base.

- Testez aussi quelques mots de langues différentes avec le modèle multilingue
(par exemple chien/dog)

In [ ]:
import torch.nn.functional as F

F.cosine_similarity()



TypeError: cosine_similarity(): argument 'x1' (position 1) must be Tensor, not str

# Visualisation

On va maintenant faire une réduction de dimension pour essayer de voir des régularités de l'espace des embeddings ("espace latent")

Nous utiliserons la méthode UMAP qui est bien adaptée en grande dimension, avec les paramètres suivants

In [84]:
# vous pouvez faire varier les 2 premiers pour voir, mais attention le calcul est assez long
params={"n_neighbors":15,
        "min_dist":0.1,
        "n_components":2,
        "metric":"euclidean",
       }

On peut appeler la méthode comme suit.

In [10]:
reducer = umap.UMAP(random_state=42,**params)
# méthode alternative plus lente
# reducer = TSNE(n_components=2, learning_rate='auto',init='random', perplexity=3)
#-------
# attention meme umap va prendre quelques minutes ! (~5mn dans colab)
%time  redux = reducer.fit_transform(initial_embed_matrix.detach().numpy())

CPU times: user 24 μs, sys: 5 μs, total: 29 μs
Wall time: 38.6 μs


NameError: name 'initial_embed_matrix' is not defined

redux est maintenant une nouvelle représentation des sous-tokens dans un espace en 2 dimensions. Vérifiez sa taille.
On va visualiser seulement les sous-tokens les plus courants pour ne pas trop surcharger.
Récupérer les indices des N=1000 (par exemple) sous-tokens les plus courants à partir du comptage fait plus haut.


In [ ]:
from numpy import random
# ce code affiche la projection en 2D, avec la chaine correspondant à chaque sous-token pour un échantillon
# au hasard
# remplacez l'échantillon par les N sous-tokens les plus courants et observez la différence
fig, ax = plt.subplots(figsize=(30, 20))
#plt.scatter(redux[:, 0], redux[:, 1],alpha=0.1)
top100_pt = redux[np.array(top100_id)]
print(top100_pt.shape)
plt.scatter(top100_pt[:,0],top100_pt[:,1],s=100,c="r",alpha=0.2)
#
sample = random.randint(redux.shape[0],size=100)
for idx in list(sample.data):
    ax.annotate(tokenizer.convert_ids_to_tokens(idx),redux[idx],size=20,c="w",weight="bold")
    ax.annotate(tokenizer.convert_ids_to_tokens(idx),redux[idx],size=20)

# Traduction / mapping entre langues

On va charger un dictionnaire bilingue avec la librairie word2word:

In [ ]:
en2fr = word2word.Word2word("en", "fr")

Pour simplifier on va juste regarder la première traduction proposée pour chaque mot anglais qui est un sous-token dans un modèle multilingue comme mBert.

- Vérifiez le contenu du mapping en2fr, prenez un exemple, et calculez le mapping sous-token anglais -> mot français qui est aussi un sous-token du modèle.
Cela nous donnera un ensemble de mot en/fr qui sont dans le vocabulaire des sous-tokens du modèle.

- On va maintenant regarder pour chaque token "anglais" quel est le token "français" le plus proche: calculez cette association avec la similarité cosinus. On peut sauver le tout dans une table pandas.

- regardez la distribution des similarités. regardez les valeurs à 1, et les valeurs élevées mais <1 (par exemple 0.5-.99)

In [ ]:
# contenu du dictionnaire ?
en2fr.__dict__.keys()

In [ ]:
# test
en2fr('militia')

## Analyse

- que peut-on dire des mots les plus proches ?

- si on veut vérifier que les traductions sont toujours proches on va reprendre la projection de l'espace latent en affichant les traductions et la distance mot-traduction dans la figure en utilisant le code suivant (doit générer la figure exemple ou approchant).

- Que concluez vous sur les embeddings du modèle considéré ?

In [ ]:
# visualisation interactive des tokens
import plotly.express as px
# remplacez par ce que vous avez trouvé comme ensemble de mots qui sont des mots anglais
# et des mots français.
# le code suppose que target a
#  - des champs x et y pour les coordonnées de la projection umap
#  - un champ pour dire dans quel langue le token est un mot
#  optionnellement: (la figure peut se faire sans mais c'est moins intéressant)
#  - un champ tok qui donne la chaine du token
#  - un champ french_word pour la traduction du mot en français (si mot pas français)
#  - un champ distance pour la distance cosinus entre le mot et sa traduction
target = "?"

fig = px.scatter(both, x="x", y="y", color="lang",
                 color_discrete_sequence=['black','red'],
                 hover_name='tok',hover_data=["french_word","distance"],
                 width=1200, height=800,
                 template="seaborn")

fig.update_traces(opacity=.1)
fig.show()


# Question Bonus:

refaire la visualisation de l'espace latent avec la projection finale des mots anglais passés dans le modèle *encodeur*

vous pouvez utiliser les fonctions suivantes qui appelent un modèle encodeur sur une liste de mot et récupérent les activations de chaque couche moyennées sur les sous-tokens d'un mot

vous pouvez le faire avec un échantillon de mots au hasard

In [ ]:
# appel du modèle + recup toutes les couches visées en les sommant
def get_hidden_states(encoded, token_ids_word, model, layers):
    # inference
    with torch.no_grad():
        output = model(**encoded,output_hidden_states=True)
    # get all hidden states
    states = output.hidden_states
    # stack and sum layers
    output = torch.stack([states[i] for i in layers]).sum(0).squeeze()
    # subset for tokens that make up the word, averaged on subtokens
    word_tokens_output = output[token_ids_word]
    return word_tokens_output.mean(dim=0)

# encode the sentence, get vector representing the word at index idx for given layers
def get_word_vector(sent, idx, tokenizer, model, layers, device):
    # tokenize the input sentence and sending to device
    encoded = tokenizer.encode_plus(sent, add_special_tokens=True, return_tensors="pt").to(device)
    # get all token idxs that make up the target word
    token_ids_word = np.where(np.array(encoded.word_ids()) == idx)
    # get all hidden states
    return get_hidden_states(encoded, token_ids_word, model, layers)

# main function to get encoding vector of given word occuring in a given sentence
# layers = set of layers to average for the representation
# if want only the last one call with layers = [-1]
def get_embedding(model, tokenizer, sent, word, device, layers=None):
    # using last four layers by default
    layers = [-4, -3, -2, -1] if layers is None else layers
    # get idx for the word
    idx = sent.split(" ").index(word)
    # get word embedding
    word_embedding = get_word_vector(sent, idx, tokenizer, model, layers, device)
    return word_embedding